# Data Science Project 1 — Advanced EDA & Feature Engineering### Dataset: Employee Data (employeeesss.xlsx)Is notebook mein hum:1. Missing values handle karenge (statistical imputation)2. Outliers neutralize karenge (IQR method)3. Kam se kam 3 nayi predictive features banayenge4. Categorical columns ko One-Hot Encode karenge5. Final clean dataset save karenge**Batch: 2026 | DecodeLabs Industrial Training Kit**

## Step 1: Upload & Load Dataset

In [13]:
from google.colab import files
uploaded = files.upload()  # yahan employeeesss.xlsx select karo

Saving employeeesss.csv to employeeesss (1).csv


In [14]:
import pandas as pd
import numpy as np
df = pd.read_csv('employeeesss.csv')
print(df.shape)
df.head()

(320, 9)


,First Name,Last Name,Email,Phone,Gender,Department,Job Title,Years Of Experience,Salary
0,Hary,Dey,joselopez0194@slingacademy.com,+1-971-533-4552x1542,male,Product,Web Developer,6.0,6000.0
1,Diane,Carter,dianecarter1345@slingacademy.com,881.633.0107,female,Human Resource,HR Manager,13.0,NaN
2,Rosy,NaN,sherryfoster2572@slingacademy.com,001-966-861-0065x493,female,Product,Tester,8.0,11000.0
3,Brenda,Fisher,brendafisher3114@slingacademy.com,001-574-564-4648,female,Product,Project Manager,14.0,17000.0
4,Sharon,Hunter,sharonhunter4898@slingacademy.com,5838355842,female,Product,Machine Learning Engineer,7.0,10500.0


## Step 2: Initial Inspection\nMissingness aur data types check karo, cleaning shuru karne se pehle.

In [15]:
print(df.info())
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   First Name           320 non-null    object 
 1   Last Name            319 non-null    object 
 2   Email                318 non-null    object 
 3   Phone                317 non-null    object 
 4   Gender               319 non-null    object 
 5   Department           318 non-null    object 
 6   Job Title            318 non-null    object 
 7   Years Of Experience  319 non-null    float64
 8   Salary               316 non-null    float64
dtypes: float64(2), object(7)
memory usage: 22.6+ KB
None

Missing values per column:
First Name             0
Last Name              1
Email                  2
Phone                  3
Gender                 1
Department             2
Job Title              2
Years Of Experience    1
Salary                 4
dtype: int64

       Years Of Exper

## Step 3: Handling Missing Values (Decision Matrix)- < 5% missing -> statistical imputation ya row drop- Salary / Years Of Experience -> median imputation (skewed data ke liye robust)- Categorical columns (Gender, Department, Job Title, Last Name) -> mode imputation- Email / Phone -> identifier columns, missing rows drop kar dete hain

In [16]:
# Numeric columns: median imputation
df['Salary'] = df['Salary'].fillna(df['Salary'].median())
df['Years Of Experience'] = df['Years Of Experience'].fillna(df['Years Of Experience'].median())

# Categorical columns: mode imputation
for col in ['Gender', 'Department', 'Job Title', 'Last Name']:
    df[col] = df[col].fillna(df[col].mode()[0])

# Identifier columns: drop rows with missing Email/Phone
df = df.dropna(subset=['Email', 'Phone']).reset_index(drop=True)

print("Missing values after cleaning:")
print(df.isnull().sum())

Missing values after cleaning:
First Name             0
Last Name              0
Email                  0
Phone                  0
Gender                 0
Department             0
Job Title              0
Years Of Experience    0
Salary                 0
dtype: int64


## Step 4: Outlier Detection & Neutralization (IQR Method)Row delete karne ke bajaye `numpy.clip()` se values ko statistical bounds par cap karte hain —isse row count aur data volume preserve rehta hai.

In [17]:
Q1 = df['Salary'].quantile(0.25)
Q3 = df['Salary'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")
outlier_count = ((df['Salary'] < lower_bound) | (df['Salary'] > upper_bound)).sum()
print(f"Outliers detected: {outlier_count}")
df['Salary'] = np.clip(df['Salary'], lower_bound, upper_bound)

Lower bound: 2500.0, Upper bound: 18500.0
Outliers detected: 0


## Step 5: Feature EngineeringKam se kam 3 nayi predictive features engineer karenge existing columns se.

In [20]:
# Feature 1: Salary per year of experience (productivity/value proxy)
df['Salary_per_Experience'] = df['Salary'] / (df['Years Of Experience'] + 1)

# Feature 2: Experience level bucket (categorical seniority feature)
df['Experience_Level'] = pd.cut(
    df['Years Of Experience'],
    bins=[0, 3, 7, 15, 100],
    labels=['Junior', 'Mid', 'Senior', 'Expert'])

# Feature 3: Full name (combined identity feature)
df['Full_Name'] = df['First Name'] + ' ' + df['Last Name']

# Feature 4 (bonus): Email domain
df['Email_Domain'] = df['Email'].str.split('@').str[1]

df[['Full_Name', 'Years Of Experience', 'Experience_Level', 'Salary', 'Salary_per_Experience', 'Email_Domain']].head()

,Full_Name,Years Of Experience,Experience_Level,Salary,Salary_per_Experience,Email_Domain
0,Hary Dey,6.0,Mid,6000.0,857.142857,slingacademy.com
1,Diane Carter,13.0,Senior,10500.0,750.000000,slingacademy.com
2,Rosy Smith,8.0,Senior,11000.0,1222.222222,slingacademy.com
3,Brenda Fisher,14.0,Senior,17000.0,1133.333333,slingacademy.com
4,Sharon Hunter,7.0,Mid,10500.0,1312.500000,slingacademy.com


## Step 6: Categorical Encoding (One-Hot Encoding)Label Encoding se bachna hai kyunki wo false ordinal relationship create karta hai.One-Hot Encoding har category ko apna independent coordinate axis deta hai.

In [22]:
df_encoded = pd.get_dummies(
    df,
    columns=['Gender', 'Department', 'Job Title', 'Experience_Level'],
    drop_first=True)
df_encoded.head()

,First Name,Last Name,Email,Phone,Years Of Experience,Salary,Salary_per_Experience,Full_Name,Email_Domain,Gender_male,...,Job Title_DevOps Engineer,Job Title_HR Manager,Job Title_Machine Learning Engineer,Job Title_Mobile Developer,Job Title_Project Manager,Job Title_Tester,Job Title_Web Developer,Experience_Level_Mid,Experience_Level_Senior,Experience_Level_Expert
0,Hary,Dey,joselopez0194@slingacademy.com,+1-971-533-4552x1542,6.0,6000.0,857.142857,Hary Dey,slingacademy.com,True,...,False,False,False,False,False,False,True,True,False,False
1,Diane,Carter,dianecarter1345@slingacademy.com,881.633.0107,13.0,10500.0,750.000000,Diane Carter,slingacademy.com,False,...,False,True,False,False,False,False,False,False,True,False
2,Rosy,Smith,sherryfoster2572@slingacademy.com,001-966-861-0065x493,8.0,11000.0,1222.222222,Rosy Smith,slingacademy.com,False,...,False,False,False,False,False,True,False,False,True,False
3,Brenda,Fisher,brendafisher3114@slingacademy.com,001-574-564-4648,14.0,17000.0,1133.333333,Brenda Fisher,slingacademy.com,False,...,False,False,False,False,True,False,False,False,True,False
4,Sharon,Hunter,sharonhunter4898@slingacademy.com,5838355842,7.0,10500.0,1312.500000,Sharon Hunter,slingacademy.com,False,...,False,False,True,False,False,False,False,True,False,False


## Step 7 (Bonus): Multicollinearity CheckNumeric features ke beech correlation check karte hain — agar koi pair 0.80+ correlated ho,to target ke saath weak correlation wala column drop kar sakte hain.

In [24]:
numeric_cols = df_encoded.select_dtypes(include=[np.number])
corr_matrix = numeric_cols.corr()
corr_matrix

,Years Of Experience,Salary,Salary_per_Experience
Years Of Experience,1.000000,0.864273,-0.785401
Salary,0.864273,1.000000,-0.535575
Salary_per_Experience,-0.785401,-0.535575,1.000000


## Step 8: Save Final Clean Dataset

In [26]:
df_encoded.to_csv('cleaned_employee_data.csv', index=False)
from google.colab import files
files.download('cleaned_employee_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ConclusionIs notebook mein humne raw employee data ko:- Missing values se saaf kiya (statistical imputation)- Outliers ko IQR method se neutralize kiya- 4 nayi predictive features engineer ki- Categorical variables ko One-Hot Encode kiya- Multicollinearity check kiYe clean dataset ab kisi bhi Machine Learning model (regression/classification) ke liye ready hai.